# Customer Churn Prediction & Retention Automation Pipeline

Architecture
------------
1. Feature Engineering
   - Behavioral features (tenure, usage frequency, support tickets, recency)
   - Financial features (monthly spend, plan tier, payment failures)
   - Engineered features (usage trend, spend-to-tenure ratio) — hand-crafted
     signals that often carry more predictive power than raw columns

2. ChurnModel
   - Gradient Boosted Trees (XGBoost) — industry-standard for tabular churn problems
     because it handles non-linear interactions and mixed feature types well,
     trains fast, and gives usable feature importances for stakeholder explanations
   - Class-imbalance handling via scale_pos_weight (churners are usually a minority class)

3. RetentionAutomationEngine
   - Consumes churn probability + customer segment
   - Applies a rules layer to decide *which* retention action to trigger and at
     what priority — this simulates the "automation pipeline" half of the system:
     in production this would publish to a queue (e.g. Kafka/SQS) consumed by a
     marketing/CRM system (send discount, assign CSM call, in-app nudge, etc.)

Why this design?
- Churn prediction alone is not actionable — pairing it with a decision/automation
  layer is what makes it a "retention" system rather than just a classifier.
- Rules are used post-model (not baked into the model) so business/marketing teams
  can iterate on retention policy without retraining the model.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, classification_report, confusion_matrix
)
from xgboost import XGBClassifier

In [2]:
# ----------------------------------------------------------------------------
# 1. Synthetic data generation (stand-in for a real customer/usage warehouse table)
# ----------------------------------------------------------------------------
def generate_synthetic_data(n_customers=8000, seed=42):
    rng = np.random.default_rng(seed)

    tenure_months = rng.integers(1, 60, size=n_customers)
    monthly_spend = rng.normal(60, 25, size=n_customers).clip(5, 300)
    plan_tier = rng.choice(["Basic", "Standard", "Premium"], size=n_customers, p=[0.5, 0.35, 0.15])
    support_tickets_last_90d = rng.poisson(1.2, size=n_customers)
    logins_last_30d = rng.poisson(8, size=n_customers)
    payment_failures_last_180d = rng.poisson(0.3, size=n_customers)
    usage_trend = rng.normal(0, 1, size=n_customers)  # negative = declining usage
    days_since_last_login = rng.exponential(10, size=n_customers).clip(0, 180)
    nps_score = rng.integers(0, 11, size=n_customers)

    # Construct a realistic latent churn probability from these features (ground truth
    # generating process — in real life this relationship is unknown and must be learned)
    logit = (
        -2.0
        - 0.03 * tenure_months
        + 0.015 * days_since_last_login
        + 0.35 * payment_failures_last_180d
        + 0.25 * support_tickets_last_90d
        - 0.4 * usage_trend
        - 0.15 * nps_score
        - 0.01 * monthly_spend
        + rng.normal(0, 0.5, size=n_customers)
    )
    churn_prob = 1 / (1 + np.exp(-logit))
    churned = (rng.random(n_customers) < churn_prob).astype(int)

    df = pd.DataFrame({
        "customer_id": np.arange(n_customers),
        "tenure_months": tenure_months,
        "monthly_spend": monthly_spend,
        "plan_tier": plan_tier,
        "support_tickets_last_90d": support_tickets_last_90d,
        "logins_last_30d": logins_last_30d,
        "payment_failures_last_180d": payment_failures_last_180d,
        "usage_trend": usage_trend,
        "days_since_last_login": days_since_last_login,
        "nps_score": nps_score,
        "churned": churned,
    })
    return df


def engineer_features(df):
    df = df.copy()
    df["spend_per_tenure_month"] = df.monthly_spend / (df.tenure_months + 1)
    df["is_high_risk_tenure"] = (df.tenure_months < 6).astype(int)  # new customers churn more
    df["low_engagement_flag"] = (df.logins_last_30d < 3).astype(int)
    df = pd.get_dummies(df, columns=["plan_tier"], drop_first=True)
    return df

In [3]:
# ----------------------------------------------------------------------------
# 2. Churn model
# ----------------------------------------------------------------------------
class ChurnModel:
    def __init__(self, seed=42):
        self.seed = seed
        self.feature_cols = None

    def fit(self, df):
        y = df["churned"]
        X = df.drop(columns=["customer_id", "churned"])
        self.feature_cols = X.columns.tolist()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=self.seed, stratify=y
        )

        # Handle class imbalance: weight the minority (churn) class up
        neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
        scale_pos_weight = neg / pos

        self.model = XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="auc",
            random_state=self.seed,
        )
        self.model.fit(X_train, y_train)

        self.X_test, self.y_test = X_test, y_test
        return self

    def evaluate(self):
        proba = self.model.predict_proba(self.X_test)[:, 1]
        auc = roc_auc_score(self.y_test, proba)
        preds = (proba >= 0.5).astype(int)

        print(f"ROC-AUC: {auc:.4f}\n")
        print("Classification report (threshold=0.5):")
        print(classification_report(self.y_test, preds, digits=3))
        print("Confusion matrix:\n", confusion_matrix(self.y_test, preds))
        return auc

    def feature_importance(self, top_n=10):
        importances = pd.Series(self.model.feature_importances_, index=self.feature_cols)
        return importances.sort_values(ascending=False).head(top_n)

    def predict_proba(self, df):
        X = df[self.feature_cols]
        return self.model.predict_proba(X)[:, 1]

In [4]:
# ----------------------------------------------------------------------------
# 3. Retention Automation Engine — turns predictions into actions
# ----------------------------------------------------------------------------
class RetentionAutomationEngine:
    """
    Rule-based decision layer sitting downstream of the churn model.
    In production this would emit events to a message queue consumed by
    marketing automation / CRM systems (e.g. Braze, Salesforce Marketing Cloud).
    """

    RULES = [
        # (min_churn_prob, min_spend, action, priority)
        (0.75, 100, "Assign dedicated Customer Success Manager call", "P0 - Critical"),
        (0.75, 0, "Send high-value retention discount (20%) + CSM email", "P1 - High"),
        (0.50, 0, "Send targeted re-engagement email + 10% discount", "P2 - Medium"),
        (0.30, 0, "Add to in-app re-engagement nudge campaign", "P3 - Low"),
    ]

    def decide_action(self, churn_prob, monthly_spend):
        for threshold, spend_threshold, action, priority in self.RULES:
            if churn_prob >= threshold and monthly_spend >= spend_threshold:
                return action, priority
        return "No action — low churn risk", "P4 - Monitor"

    def run(self, df, churn_probs):
        results = []
        for prob, spend, cust_id in zip(churn_probs, df.monthly_spend, df.customer_id):
            action, priority = self.decide_action(prob, spend)
            results.append((cust_id, prob, spend, action, priority))
        return pd.DataFrame(
            results, columns=["customer_id", "churn_probability", "monthly_spend", "action", "priority"]
        ).sort_values("churn_probability", ascending=False)


if __name__ == "__main__":
    raw_df = generate_synthetic_data(n_customers=8000)
    features_df = engineer_features(raw_df)

    model = ChurnModel().fit(features_df)
    print("=== Model Evaluation ===")
    model.evaluate()

    print("\n=== Top 10 Predictive Features ===")
    print(model.feature_importance(10))

    # Simulate scoring the full active customer base + triggering retention actions
    churn_probs = model.predict_proba(features_df)
    engine = RetentionAutomationEngine()
    action_plan = engine.run(features_df.assign(monthly_spend=raw_df.monthly_spend), churn_probs)

    print("\n=== Sample Retention Action Plan (Top 10 highest-risk customers) ===")
    print(action_plan.head(10).to_string(index=False))

    print("\n=== Action Distribution Across Customer Base ===")
    print(action_plan.priority.value_counts())

=== Model Evaluation ===
ROC-AUC: 0.6837

Classification report (threshold=0.5):
              precision    recall  f1-score   support

           0      0.968     0.874     0.918      1534
           1      0.098     0.318     0.150        66

    accuracy                          0.851      1600
   macro avg      0.533     0.596     0.534      1600
weighted avg      0.932     0.851     0.887      1600

Confusion matrix:
 [[1341  193]
 [  45   21]]

=== Top 10 Predictive Features ===
nps_score                     0.118673
tenure_months                 0.107098
usage_trend                   0.095829
payment_failures_last_180d    0.081104
days_since_last_login         0.077522
spend_per_tenure_month        0.072466
monthly_spend                 0.071532
plan_tier_Standard            0.068690
plan_tier_Premium             0.066445
support_tickets_last_90d      0.064573
dtype: float32

=== Sample Retention Action Plan (Top 10 highest-risk customers) ===
 customer_id  churn_probability  mo